In [2]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm.notebook import tqdm
from sklearn.linear_model import Ridge, RidgeCV

In [3]:
adata = sc.read_h5ad("/home/wergillius/Project/diffuse_differentiate/data/Barcodelet/integrated_mesc_group0_Nov7.h5ad")
adata

AnnData object with n_obs × n_vars = 5745 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh', 'assignment', 'starting.state', 'dataset', 'control', 'cell_type', 'condition', 'dose_val', 'batch', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'dpt_pseudotime', 'RA+Wnt+Fgf', 'numeric_iloc', 'split', 'discrete_time'
    var: 'gene_name'
    uns: 'condition_colors', 'diff', 'diffmap_evals', 'dpt_changepoints', 'dpt_groups_colors', 'dpt_grouptips', 'iroot', 'neighbors', 'umap', 'unique_token_dict'
    obsm: 'Tr_SampledX_r100', 'X_diffmap', 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'diff_connectivities', 'diff_distances', 'distances'

# ridge prior

In [4]:
perturbations = ['RA', 'Wnt', 'TgfB', 'Bmp', 'Fgf', 'Notch', 'Shh']
assignment_C = adata.obs[perturbations].values
gene_X = adata.X

X =  np.concatenate([gene_X, assignment_C], axis=1)

Y = adata.obsm['Tr_SampledX_r100'].copy()

train_idx = adata.obs.split != 'test'
test_idx = adata.obs.split == 'test'

x_train = X[train_idx]
x_test = X[test_idx]

Y_train = Y[train_idx]
Y_test = Y[test_idx]

In [5]:
N_gene = Y.shape[1]
N_gene

2000

In [6]:
def fit_gene_i(gene_idx):
    """

    Args:
        gene_idx (_type_): _description_

    Returns: tuple: tuple of (Weight, intercetpt, r2)
    Weight : np.array (2007,), \beta_i , 
    intercept : float, beta_0
    r2 : performance on test set
    """
    W_i = RidgeCV().fit(x_train, Y_train[:, gene_idx])
    r2 = W_i.score(x_test, Y_test[:,gene_idx])
    return W_i.coef_, W_i.intercept_, r2

In [7]:
from tqdm.contrib.concurrent import process_map

In [9]:
priors = process_map(fit_gene_i, np.arange(N_gene),  # fn & iters
                     max_workers=30, chunksize=10)   # multi-process kwargs

  0%|          | 0/2000 [00:00<?, ?it/s]

In [21]:
# Priors = np.stack(priors)
Coef_Mat = np.stack([p[0] for p in priors])
Intcpt_Mat = np.stack([p[1] for p in priors])
r2_Mat = np.stack([p[2] for p in priors])

In [22]:
Coef_Mat.shape

(2000, 2007)

In [23]:
Intcpt_Mat.shape

(2000,)

In [24]:
save_dir = "/home/wergillius/Project/diffuse_differentiate/result/Barcodelet"
np.save(
    f"{save_dir}/prior_coef_Jul2.npy",
    Coef_Mat
)
np.save(
    f"{save_dir}/prior_intcpt_Jul2.npy",
    Intcpt_Mat
)

pretrain to new checkpoint